# Stg Layer — search_results (Incremental)

Read **search_results** từ Lakebase Postgres (append-only discovery log — 1 row / (video, keyword, search run), không update) → MERGE 1:1 vào Delta staging table. Dùng MERGE trên `id` (thay vì INSERT thuần) để re-run sau lỗi vẫn idempotent, dù nguồn không bao giờ update rows.

**Incremental Flow:**
1. Lấy watermark (`MAX(fetched_at)`) từ Delta table
2. SELECT từ `lakebase_pg.public.search_results` (foreign catalog): chỉ rows có `fetched_at > watermark`
3. MERGE vào `prod.default.stg_search_results` (UPSERT on `id`)
4. Verify row count

**Kiến trúc:** Databricks App → user search/save YouTube data → INSERT/UPDATE vào Postgres → Job notebook (this) → SELECT Postgres via foreign catalog (incremental) → MERGE vào Delta (stg layer, 1:1 với nguồn).

> **Note:** Đọc qua foreign catalog `lakebase_pg` thay vì JDBC để tránh timezone mismatch giữa Spark (UTC) và Postgres — cùng pattern với `Analytic/landing/YouTube Ingest — Lakebase Postgres.ipynb`.

In [0]:
# --- Config ---
PG_TABLE = "lakebase_pg.public.search_results"  # Lakebase Postgres foreign catalog
DELTA_TABLE = "prod.default.stg_search_results"
MERGE_KEY = "id"

print(f"Postgres source: {PG_TABLE}")
print(f"Delta target: {DELTA_TABLE}")

In [0]:
# --- Get watermark from Delta table (last successful ETL) ---
try:
    watermark_row = spark.sql(f"SELECT MAX(fetched_at) AS wm FROM {DELTA_TABLE}").collect()[0]
    watermark = watermark_row["wm"] if watermark_row else None
except Exception:
    watermark = None

if watermark:
    # Incremental: only rows changed after watermark
    # Use Spark SQL — timestamps stay in Spark's timezone (UTC), no JDBC mismatch
    pg_df = spark.sql(f"""
        SELECT * FROM {PG_TABLE}
        WHERE fetched_at > '{watermark.strftime("%Y-%m-%d %H:%M:%S.%f")}'
    """)
    print(f"💧 Watermark: {watermark}")
    print("Incremental load: rows after watermark")
else:
    # Full load (first run or Delta empty)
    pg_df = spark.sql(f"SELECT * FROM {PG_TABLE}")
    print("📦 Full load (no watermark — first run)")

pg_count = pg_df.count()
print(f"Read {pg_count} rows from Lakebase Postgres ✅")
if pg_count > 0:
    display(pg_df.limit(10))

In [0]:
# --- Add etl_date and op_type to source data ---
from pyspark.sql.functions import current_timestamp, lit
pg_df = pg_df.withColumn("etl_date", current_timestamp())
pg_df = pg_df.withColumn("op_type", lit("INSERT"))

# --- Create Delta table if not exists (1:1 with source + ETL metadata) ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DELTA_TABLE} (
        id             BIGINT,
        video_id       STRING,
        channel_id     STRING,
        title          STRING,
        published_at   TIMESTAMP,
        search_keyword STRING,
        fetched_at     TIMESTAMP,
        etl_date       TIMESTAMP,
        op_type        STRING
    )
    USING DELTA
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"Delta table {DELTA_TABLE} ready ✅")

# --- MERGE via SQL (compact, op_type tracking) ---
pg_df.createOrReplaceTempView("pg_source")

spark.sql(f"""
    MERGE INTO {DELTA_TABLE} AS t
    USING pg_source AS s ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET
        video_id=s.video_id,
        channel_id=s.channel_id,
        title=s.title,
        published_at=s.published_at,
        search_keyword=s.search_keyword,
        fetched_at=s.fetched_at,
        etl_date=s.etl_date, op_type='UPDATE'
    WHEN NOT MATCHED THEN INSERT *
""")

merged_count = spark.sql(f"SELECT COUNT(*) FROM {DELTA_TABLE}").collect()[0][0]
print(f"✅ Merged into {DELTA_TABLE}: {merged_count} total rows")

In [0]:
# --- Verify ---
total_delta = spark.sql(f"SELECT COUNT(*) as cnt FROM {DELTA_TABLE}").collect()[0]["cnt"]
last_etl = spark.sql(f"SELECT MAX(etl_date) as etl FROM {DELTA_TABLE}").collect()[0]["etl"]

op_counts = spark.sql(f"SELECT op_type, COUNT(*) as cnt FROM {DELTA_TABLE} GROUP BY op_type").collect()
op_breakdown = ", ".join(f"{r['op_type']}: {r['cnt']}" for r in op_counts)

print(f"📊 Total rows in {DELTA_TABLE}: {total_delta}")
print(f"📊 Last ETL date: {last_etl}")
print(f"📊 Op type breakdown: {op_breakdown}")
print(f"📊 Rows loaded this run: {pg_count}")
print("\n--- stg_search_results ETL complete ---")